In [ ]:
using WignerD

"""
1ピクセル分の離散平均 D̄^l_{mn} を direct に計算する基準版。

D^l_{mn}(α,β,γ) = exp(-imα) * d^l_{mn}(β) * exp(-inγ)

入力:
- alphas, betas, gammas :: AbstractVector{<:Real}
- lmax :: Int
- nmax :: Int = 10

返り値:
- Davg :: Vector{Matrix{ComplexF64}}
  Davg[l+1] は size = (2l+1, 2nmax_eff+1), nmax_eff = min(nmax, l)
  行: m=-l:l, 列: n=-nmax_eff:nmax_eff

補助関数:
- row index: im = m + l + 1
- col index: in = n + nmax_eff + 1
"""
function mean_Dlmns_direct_onepixel(
    alphas::AbstractVector{<:Real},
    betas::AbstractVector{<:Real},
    gammas::AbstractVector{<:Real};
    lmax::Int,
    nmax::Int = 10,
)
    # --- checks ---
    N = length(alphas)
    N == length(betas) == length(gammas) || throw(ArgumentError("alphas, betas, gammas must have the same length"))
    N > 0 || throw(ArgumentError("input angle arrays must be non-empty"))
    lmax >= 0 || throw(ArgumentError("lmax must be non-negative"))
    nmax >= 0 || throw(ArgumentError("nmax must be non-negative"))

    # 出力コンテナ（ℓごとにサイズが違う）
    Davg = Vector{Matrix{ComplexF64}}(undef, lmax + 1)

    for l in 0:lmax
        nmax_eff = min(nmax, l)
        Davg[l+1] = zeros(ComplexF64, 2l + 1, 2nmax_eff + 1)
    end

    invN = 1.0 / N

    # ---------------------------------------------------------
    # サンプルごとに寄与を加算
    # ---------------------------------------------------------
    @inbounds for j in eachindex(alphas)
        α = float(alphas[j])
        β = float(betas[j])
        γ = float(gammas[j])

        # 各 l ごとに処理
        for l in 0:lmax
            nmax_eff = min(nmax, l)
            M = Davg[l+1]

            # 位相を毎回 exp でも基準版としてはOK
            # （後で高速化版ではここを再利用/漸化式に置換）
            for n in -nmax_eff:nmax_eff
                phase_n = cis(-n * γ)  # exp(-i n γ)

                for m in -l:l
                    phase_m = cis(-m * α)  # exp(-i m α)

                    dmn = WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0)
                    # dmn は通常実数だが、型に依存して複素かもしれないのでそのまま扱う
                    Dmn = phase_m * dmn * phase_n

                    im = m + l + 1
                    in_idx = n + nmax_eff + 1
                    M[im, in_idx] += Dmn
                end
            end
        end
    end

    # 平均を取る
    for l in 0:lmax
        Davg[l+1] .*= invN
    end

    return Davg
end

mean_Dlmns_direct_onepixel

In [2]:
# 例: 1ピクセル分の角度列（ダミー）
N = 500
alphas = 2π .* rand(N)
betas  = 1e-3 .* rand(N)         # 小β
gammas = 2π .* rand(N)

lmax = 20
nmax = 10

Davg = mean_Dlmns_direct_onepixel(alphas, betas, gammas; lmax=lmax, nmax=nmax)

# 例: l=5 の行列サイズ確認
@show size(Davg[5+1])  # (2*5+1, 2*min(10,5)+1) = (11, 11)

# 例: l=5, m=2, n=-1 の平均値を読む
l = 5
m = 2
n = -1
nmax_eff = min(nmax, l)
im = m + l + 1
in_idx = n + nmax_eff + 1
@show Davg[l+1][im, in_idx]

size(Davg[5 + 1]) = (11, 11)
(Davg[l + 1])[im, in_idx] = -4.374670382330551e-11 - 7.294702812179673e-12im


-4.374670382330551e-11 - 7.294702812179673e-12im

In [3]:
# 例: l=5, m=2, n=-1 の平均値を読む
l = 5
m = 2
n = -1
nmax_eff = min(nmax, l)
im = m + l + 1
in_idx = n + nmax_eff + 1
@show Davg[l+1][im, in_idx]

(Davg[l + 1])[im, in_idx] = -4.374670382330551e-11 - 7.294702812179673e-12im


-4.374670382330551e-11 - 7.294702812179673e-12im

In [4]:
l=20
Davg[l+1]

41×21 Matrix{ComplexF64}:
  2.21018e-19+2.39576e-17im  …  -2.03672e-17-9.97748e-18im
 -1.50095e-20-1.32025e-20im     -8.82233e-20-3.31954e-19im
 -3.21527e-17+1.44189e-18im     -5.17033e-17+2.9642e-17im
  -6.3622e-20-1.38648e-19im     -1.20073e-19-1.14376e-19im
  4.35217e-18-1.02462e-17im     -7.78955e-18-1.3575e-19im
 -2.81472e-15+1.58805e-15im  …   9.56733e-20-3.52655e-19im
 -1.91781e-12-1.52819e-12im      2.64797e-18+1.3748e-17im
   1.02578e-9-3.73875e-10im      7.58038e-19+4.31733e-19im
  -6.40274e-7-6.275e-7im        -1.02451e-17+3.7435e-17im
  -0.00012684-7.97666e-6im       1.10931e-19+3.37241e-20im
             ⋮               ⋱              ⋮
  5.51833e-18+2.15819e-17im      -6.40274e-7+6.275e-7im
  6.26236e-19-3.60379e-19im      -1.02578e-9-3.73875e-10im
 -5.58605e-18+2.27807e-17im     -1.91805e-12+1.5284e-12im
   7.9138e-21+1.68772e-20im  …   2.81569e-15+1.58951e-15im
 -3.17315e-17+1.97453e-19im     -2.51119e-17-2.84605e-17im
 -3.75976e-19+3.56544e-19im     -6.56669e-20+1.1258

In [5]:
using WignerD
using SparseArrays

# ------------------------------------------------------------
# （再掲）blocks -> block diagonal sparse
# ------------------------------------------------------------
function blocks_to_blockdiag_sparse(blocks::Vector{<:AbstractMatrix})
    L = length(blocks)
    nrows_each = [size(B, 1) for B in blocks]
    ncols_each = [size(B, 2) for B in blocks]

    row_offsets = Vector{Int}(undef, L)
    col_offsets = Vector{Int}(undef, L)

    r0 = 1
    c0 = 1
    for i in 1:L
        row_offsets[i] = r0
        col_offsets[i] = c0
        r0 += nrows_each[i]
        c0 += ncols_each[i]
    end

    nrow_total = sum(nrows_each)
    ncol_total = sum(ncols_each)

    # COOで詰める
    total_nnz_est = sum(length, blocks)
    I = Int[]; J = Int[]; V = ComplexF64[]
    sizehint!(I, total_nnz_est)
    sizehint!(J, total_nnz_est)
    sizehint!(V, total_nnz_est)

    for i in 1:L
        B = blocks[i]
        ro = row_offsets[i] - 1
        co = col_offsets[i] - 1
        for col in axes(B, 2), row in axes(B, 1)
            v = B[row, col]
            if v != 0.0 + 0.0im
                push!(I, ro + row)
                push!(J, co + col)
                push!(V, ComplexF64(v))
            end
        end
    end

    S = sparse(I, J, V, nrow_total, ncol_total)
    return S, row_offsets, col_offsets
end

# ------------------------------------------------------------
# βビン情報を作る
# ------------------------------------------------------------
"""
βでビン分けし、各ビンに属するサンプル index と代表β（ビン内平均）を返す。

返り値:
- bin_indices :: Vector{Vector{Int}}   # 各ビンに入った元サンプル index
- beta_reps   :: Vector{Float64}       # 各ビンの代表β（ビン内平均）
- counts      :: Vector{Int}
"""
function make_beta_bins(
    betas::AbstractVector{<:Real};
    nbins::Int = 200,
)
    N = length(betas)
    N > 0 || throw(ArgumentError("betas must be non-empty"))
    nbins >= 1 || throw(ArgumentError("nbins must be >= 1"))

    βf = Float64.(betas)
    βmin = minimum(βf)
    βmax = maximum(βf)

    # 全βが同じなら1ビン
    if βmax == βmin
        return [collect(1:N)], [βmin], [N]
    end

    counts = zeros(Int, nbins)
    sumsβ  = zeros(Float64, nbins)
    bin_indices = [Int[] for _ in 1:nbins]

    invw = nbins / (βmax - βmin)

    @inbounds for j in eachindex(βf)
        β = βf[j]
        k = clamp(Int(floor((β - βmin) * invw)) + 1, 1, nbins)
        push!(bin_indices[k], j)
        counts[k] += 1
        sumsβ[k] += β
    end

    # 空ビンを落とす
    bin_indices_nz = Vector{Vector{Int}}()
    beta_reps = Float64[]
    counts_nz = Int[]
    sizehint!(bin_indices_nz, nbins)
    sizehint!(beta_reps, nbins)
    sizehint!(counts_nz, nbins)

    for k in 1:nbins
        c = counts[k]
        if c > 0
            push!(bin_indices_nz, bin_indices[k])
            push!(beta_reps, sumsβ[k] / c)  # ビン内平均β
            push!(counts_nz, c)
        end
    end

    return bin_indices_nz, beta_reps, counts_nz
end

# ------------------------------------------------------------
# 1ピクセル用 β-binned 近似版（blocks返し）
# ------------------------------------------------------------
"""
1ピクセル分の離散平均 D̄^l_{mn} を β-binned 近似で計算する。

近似:
  βをビン分けし、各ビン内で d^l_{mn}(β_j) ≈ d^l_{mn}(β_rep[k]) とみなす

返り値:
- blocks :: Vector{Matrix{ComplexF64}}
  blocks[l+1] は size = (2l+1, 2nmax_eff+1), nmax_eff = min(nmax,l)
"""
function mean_Dlmns_binned_onepixel_blocks(
    alphas::AbstractVector{<:Real},
    betas::AbstractVector{<:Real},
    gammas::AbstractVector{<:Real};
    lmax::Int,
    nmax::Int = 10,
    nbins::Int = 200,
)
    # --- checks ---
    N = length(alphas)
    N == length(betas) == length(gammas) || throw(ArgumentError("alphas, betas, gammas must have the same length"))
    N > 0 || throw(ArgumentError("input angle arrays must be non-empty"))
    lmax >= 0 || throw(ArgumentError("lmax must be non-negative"))
    nmax >= 0 || throw(ArgumentError("nmax must be non-negative"))

    # 出力ブロック初期化
    blocks = Vector{Matrix{ComplexF64}}(undef, lmax + 1)
    for l in 0:lmax
        nmax_eff = min(nmax, l)
        blocks[l+1] = zeros(ComplexF64, 2l + 1, 2nmax_eff + 1)
    end

    # βビン構築
    bin_indices, beta_reps, _counts = make_beta_bins(betas; nbins=nbins)

    # 各ビンごとに寄与を加算
    @inbounds for kb in eachindex(beta_reps)
        idxs = bin_indices[kb]
        βrep = beta_reps[kb]

        # ---- このビン内で、位相和 S_l[m,n] = Σ_j exp(-imα_j)exp(-inγ_j) を作る ----
        # lごとにサイズが違うので、lごとに別配列を持つ
        phase_sums = Vector{Matrix{ComplexF64}}(undef, lmax + 1)
        for l in 0:lmax
            nmax_eff = min(nmax, l)
            phase_sums[l+1] = zeros(ComplexF64, 2l + 1, 2nmax_eff + 1)
        end

        # ビン内サンプルの位相和を作る（dはまだ掛けない）
        for j in idxs
            α = float(alphas[j])
            γ = float(gammas[j])

            for l in 0:lmax
                nmax_eff = min(nmax, l)
                P = phase_sums[l+1]

                for n in -nmax_eff:nmax_eff
                    phase_n = cis(-n * γ)
                    in_idx = n + nmax_eff + 1

                    for m in -l:l
                        phase_m = cis(-m * α)
                        im = m + l + 1
                        P[im, in_idx] += phase_m * phase_n
                    end
                end
            end
        end

        # ---- βrepで d^l_{mn}(βrep) を評価して phase_sums に掛ける ----
        for l in 0:lmax
            nmax_eff = min(nmax, l)
            B = blocks[l+1]
            P = phase_sums[l+1]

            for n in -nmax_eff:nmax_eff
                in_idx = n + nmax_eff + 1
                for m in -l:l
                    im = m + l + 1
                    dmn = WignerD.wignerDjmn(l, m, n, 0.0, βrep, 0.0)
                    B[im, in_idx] += dmn * P[im, in_idx]
                end
            end
        end
    end

    # 平均化
    invN = 1.0 / N
    for l in 0:lmax
        blocks[l+1] .*= invN
    end

    return blocks
end

# ------------------------------------------------------------
# block sparse 返しのラッパー
# ------------------------------------------------------------
"""
β-binned 近似版を block diagonal sparse 行列で返す。
返り値:
- S, blocks, row_offsets, col_offsets
"""
function mean_Dlmns_binned_onepixel_blocksparse(
    alphas::AbstractVector{<:Real},
    betas::AbstractVector{<:Real},
    gammas::AbstractVector{<:Real};
    lmax::Int,
    nmax::Int = 10,
    nbins::Int = 200,
)
    blocks = mean_Dlmns_binned_onepixel_blocks(alphas, betas, gammas;
                                               lmax=lmax, nmax=nmax, nbins=nbins)
    S, row_offsets, col_offsets = blocks_to_blockdiag_sparse(blocks)
    return S, blocks, row_offsets, col_offsets
end

mean_Dlmns_binned_onepixel_blocksparse

In [ ]:
# ダミー入力
N = 1000
alphas = 2π .* rand(N)
betas  = 2e-3 .* rand(N)   # 小β
gammas = 2π .* rand(N)

lmax = 20
nmax = 20

# direct基準版（前に作った関数）
blocks_direct = @time mean_Dlmns_direct_onepixel(alphas, betas, gammas; lmax=lmax, nmax=nmax)

# binned版
blocks_binned = @time mean_Dlmns_binned_onepixel_blocks(alphas, betas, gammas;
                                                  lmax=lmax, nmax=nmax, nbins=100)
#=
# 例: ブロックごとの相対誤差を確認
for l in 0:lmax
    Bd = blocks_direct[l+1]
    Bb = blocks_binned[l+1]
    num = norm(Bb .- Bd)
    den = norm(Bd) + 1e-300
    println("l=$l  rel_block_err = ", num/den)
end

# 全体block sparseで比較
S_direct, ro_d, co_d = blocks_to_blockdiag_sparse(blocks_direct)
S_binned, ro_b, co_b = blocks_to_blockdiag_sparse(blocks_binned)

@show norm(S_binned - S_direct) / (norm(S_direct) + 1e-300)
=#

In [ ]:
maximum(abs.(blocks_binned.-blocks_direct))

In [42]:
blocks_binned

21-element Vector{Matrix{ComplexF64}}:
 [1.0 + 0.0im;;]
 [-0.018749015717053914 - 0.014375317272171895im -2.5244945563571873e-5 - 2.4157957438381947e-6im 9.695267918821439e-9 - 3.999695071399931e-9im; -2.0598281342111895e-5 - 1.381839992242254e-5im 0.999999363218219 + 0.0im 2.059828134211184e-5 - 1.3818399922422505e-5im; 9.695267918821439e-9 + 3.999695071399931e-9im 2.5244945563571832e-5 - 2.415795743838183e-6im -0.01874901571705389 + 0.014375317272171898im]
 [0.004780304987230343 + 0.019624827890496736im 4.854908503620415e-6 + 6.567025568214536e-6im … -7.841363044856989e-12 + 1.526061908612579e-11im -4.82035543635815e-16 - 1.1009676731738943e-14im; 3.3616664189622e-5 + 1.725912419191662e-5im -0.01874899934579992 - 0.014375319375707076im … 2.9085773155659993e-8 - 1.1999075195223753e-8im 3.560980132904144e-12 - 1.766075114395408e-11im; … ; -3.5609801333348194e-12 - 1.766075114504384e-11im 2.9085773155659993e-8 + 1.1999075195223753e-8im … -0.01874899934579992 + 0.01437531937570707im -3.3

In [44]:
blocks_direct

21-element Vector{Matrix{ComplexF64}}:
 [1.0 + 0.0im;;]
 [-0.018749015710926784 - 0.014375317276746418im -2.524619581093383e-5 - 2.403371914708536e-6im 9.702774626465912e-9 - 4.0170674532175155e-9im; -2.059166734090724e-5 - 1.3825360569010009e-5im 0.9999993632178629 + 0.0im 2.0591667340907188e-5 - 1.3825360569009948e-5im; 9.702774626465912e-9 + 4.0170674532175155e-9im 2.524619581093379e-5 - 2.4033719147085262e-6im -0.018749015710926773 + 0.014375317276746388im]
 [0.0047803050025148645 + 0.01962482791279553im 4.850882711869714e-6 + 6.563446931732121e-6im … -7.827897488540476e-12 + 1.525574277963588e-11im -4.831661316650298e-16 - 1.1016950362179618e-14im; 3.360017122484379e-5 + 1.7285681251259138e-5im -0.018748999315164466 - 0.014375319398579507im … 2.9108293226789064e-8 - 1.2051192254548616e-8im 3.5685731446220422e-12 - 1.762631831131088e-11im; … ; -3.568573144983533e-12 - 1.7626318312336458e-11im 2.9108293226789064e-8 + 1.2051192254548616e-8im … -0.018748999315164484 + 0.01437531939857

In [40]:
blocks_binned[l+1]

41×21 Matrix{ComplexF64}:
  9.01275e-18-6.221e-18im    …  -1.87816e-18-6.8232e-18im
   2.4124e-21-1.49056e-19im     -2.95268e-19+1.45602e-19im
 -6.37381e-18+1.63026e-18im     -5.82055e-18+3.87467e-17im
 -3.07393e-19+1.05509e-19im      1.32304e-19-5.58128e-20im
 -1.40084e-18+1.97077e-17im     -2.00096e-18-2.84694e-18im
  6.84544e-14-4.90922e-14im  …   7.91445e-20-4.54107e-20im
  -1.7117e-11+8.70792e-12im      1.71573e-18-1.07736e-17im
  -5.03923e-9-1.16487e-8im       7.98242e-19+8.91476e-19im
  -7.85043e-7+1.18557e-6im       1.23917e-16+9.63346e-18im
  -0.00052086-0.000249404im     -4.67879e-19-5.86359e-20im
             ⋮               ⋱              ⋮
 -7.18157e-17+5.33313e-18im      -7.85043e-7-1.18557e-6im
  6.58869e-19-7.38965e-19im       5.03923e-9-1.16487e-8im
 -2.59177e-18-1.79475e-17im      -1.7117e-11-8.70801e-12im
   3.4061e-21+3.77483e-21im  …   -6.8456e-14-4.90931e-14im
  -8.2918e-18+1.3565e-17im       3.78798e-18+8.3807e-18im
  4.06021e-19+1.45337e-19im      9.15128e-20-1.

In [1]:
using WignerD
using SpecialFunctions

# a_m = 2 / sqrt((l-m)(l+m+1))
@inline function a_coeff(l::Int, m::Int)
    return 2.0 / sqrt((l - m) * (l + m + 1))
end

# λ_m を半角で（キャンセル低減）
@inline function lambda_halfangle(m::Int, n::Int, sh::Float64, ch::Float64)
    sinβ = 2 * sh * ch
    # (n - m cosβ) / sinβ を (n-m + 2m sin^2(β/2)) / sinβ で計算
    return ((n - m) + 2m * (sh * sh)) / sinβ
end

"""
単一 (l, n, beta) で d^l_{m n}(beta) を m=0..l について計算する（安定版）。

返り値:
- x :: Vector{Float64}    # 正規化された値（各mで同一スケール下）
- logS :: Vector{Float64} # 真値は d[m] = x[m+1] * exp(logS[m+1])
  (expが溢れる可能性があるので、必要なら後述の to_float64 を使う)

実装の要点:
- m=l から m↓ へ再帰（decreasing order）  [oai_citation:4‡arXiv](https://arxiv.org/html/2311.14670v3)
- 毎ステップ必ず s=abs(new) で正規化し、logS に足す（running renormalisation）  [oai_citation:5‡arXiv](https://arxiv.org/html/2311.14670v3)
"""
function d_lmn_mdown_running_renorm(l::Int, n::Int, beta::Real;
    eps_sin::Real = 1e-15,
    seed_direct::Bool = true,  # seedだけはWignerD.jlで取る（精度重視）
)
    (l >= 0) || throw(ArgumentError("l must be >= 0"))
    (abs(n) <= l) || throw(ArgumentError("|n| <= l"))

    β = float(beta)
    sh = sin(β/2)
    ch = cos(β/2)
    sinβ = 2 * sh * ch

    # β≈0 は再帰係数が危ないので direct に逃がす（論文の極処理の考え方） [oai_citation:6‡arXiv](https://arxiv.org/html/2311.14670v3)
    if abs(sinβ) <= eps_sin
        x = zeros(Float64, l+1)
        logS = fill(-Inf, l+1)
        for m in 0:l
            d = real(WignerD.wignerDjmn(l, m, n, 0.0, β, 0.0))
            if d == 0.0
                x[m+1] = 0.0
                logS[m+1] = -Inf
            else
                x[m+1] = sign(d)
                logS[m+1] = log(abs(d))
            end
        end
        return x, logS
    end

    # --- seed: d_{l,n} ---
    d_ln = seed_direct ?
        real(WignerD.wignerDjmn(l, l, n, 0.0, β, 0.0)) :
        real(WignerD.wignerDjmn(l, l, n, 0.0, β, 0.0))  # ここは必要なら閉形式seedに差し替え可

    x = zeros(Float64, l+1)        # x[m+1] corresponds to m=0..l (we'll fill after)
    logS = fill(0.0, l+1)

    # 共有スケール：true d = x_scaled * exp(logScale)
    if d_ln == 0.0
        # seedが0なら全て0（通常は起きにくいが安全のため）
        return x, fill(-Inf, l+1)
    end

    logScale = log(abs(d_ln))
    x_m   = sign(d_ln) * 1.0       # scaled d_l
    x_mp1 = 0.0                    # scaled d_{l+1}=0

    # m=l の保存
    x[l+1] = x_m
    logS[l+1] = logScale

    # --- m=l から m=0 へ ---
    for m in l:-1:1
        am1 = a_coeff(l, m-1)
        λm  = lambda_halfangle(m, n, sh, ch)

        if m == l
            # d_{l-1} = λ_l a_{l-1} d_l   （d_{l+1}=0）
            x_mm1 = (λm * am1) * x_m
        else
            am = a_coeff(l, m)
            # d_{m-1} = λ_m a_{m-1} d_m - (a_{m-1}/a_m) d_{m+1}
            x_mm1 = (λm * am1) * x_m - (am1 / am) * x_mp1
        end

        # --- running renormalisation（毎ステップ） [oai_citation:7‡arXiv](https://arxiv.org/html/2311.14670v3) ---
        s = abs(x_mm1)
        if s == 0.0
            # 以降は0（スケールは-Inf扱い）
            x[m] = 0.0
            logS[m] = -Inf
            # 次の状態も0にして継続
            x_mp1 = 0.0
            x_m = 0.0
            logScale = -Inf
        else
            logScale += log(s)
            # 全状態を同じ s で割り、共有スケールを更新
            x_mm1 /= s
            x_m   /= s
            x_mp1 /= s

            # 保存（m-1 に相当する位置）
            x[m] = x_mm1
            logS[m] = logScale

            # 状態更新
            x_mp1 = x_m
            x_m   = x_mm1
        end
    end

    return x, logS
end

"""
x,logS から Float64 に復元（オーバーフロー/アンダーフロー回避のため base-2 で復元）
"""
function to_float64(x::Float64, logS::Float64)
    if !isfinite(logS) || x == 0.0
        return 0.0
    end
    # exp(logS) を 2^e * mant で安全に復元
    log2S = logS / log(2.0)
    e = floor(Int, log2S)
    mant = exp(logS - e * log(2.0))   # mant in (0,2)
    return (x * mant) * ldexp(1.0, e)
end

to_float64

In [3]:
l = 100
n = 20
beta = 1e-4

x, logS = d_lmn_mdown_running_renorm(l, n, beta)

# m=0..l の真値を取りたい場合（溢れない範囲で）
d = [to_float64(x[m+1], logS[m+1]) for m in 0:l]

@show d[1] d[n+1] d[end]   # m=0, m=n, m=l

d[1] = Inf
d[n + 1] = 5.119647111383344e300
d[end] = 1.7188356825967273e-15


1.7188356825967273e-15

In [5]:
using WignerD

"""
固定 n について d^l_{m,n}(β) を m=0..l で返す（安定版）
手順:
1) 対称性 d_{m,n}=(-1)^(m-n) d_{n,m} を使い、まず d_{n,m} を m=0..l で作る
2) そのために、conviqt/Price&McEwen系の「第2添字(order)を減らす安定再帰」を ratio で回す
3) pivot（山付近）から両側に復元（端seedから復元しない）
4) 1点（m=n）を direct 値でスケール合わせして“真の値”にする
"""
function d_fixedn_stable(l::Int, n::Int, beta::Real;
    eps_sin::Real = 1e-15
)
    (l >= 0) || throw(ArgumentError("l>=0"))
    (abs(n) <= l) || throw(ArgumentError("|n|<=l"))
    β = float(beta)

    sh = sin(β/2); ch = cos(β/2)
    sinβ = 2sh*ch
    cosβ = ch*ch - sh*sh

    # β≈0 は極：d(0)=δ_{mn}
    if abs(sinβ) <= eps_sin
        out = zeros(Float64, l+1)
        for m in 0:l
            out[m+1] = (m == n) ? 1.0 : 0.0
        end
        return out
    end

    mfix = n  # まず d_{mfix, m0} を m0=0..l で作る（ここで mfix=n）

    # ratio r[m0] = d_{mfix, m0+1} / d_{mfix, m0}
    r = zeros(Float64, l+1)
    r[l+1] = 0.0  # d_{mfix,l+1}=0 相当

    # stable recursion in m0 decreasing: m0=l..1
    for m0 in l:-1:1
        t  = (-mfix + m0*cosβ) / sinβ
        α  = 0.5 * sqrt((l + m0) * (l - m0 + 1))
        βc = 0.5 * sqrt((l - m0) * (l + m0 + 1))

        # q = d_{m0-1}/d_{m0}
        q = (t - βc*r[m0+1]) / α
        r[m0] = 1.0 / q   # r_{m0-1} = d_{m0}/d_{m0-1}
    end

    # pivot: |r|≈1 の近く（山付近）
    pivot = 0
    best = Inf
    for m0 in 1:l
        v = abs(log(abs(r[m0]) + 1e-300))
        if v < best
            best = v
            pivot = m0
        end
    end
    pivot = clamp(pivot, 0, l)

    # 形を復元（任意スケール）
    d_nm0 = zeros(Float64, l+1)   # d_{n,m0}
    d_nm0[pivot+1] = 1.0

    # 下側: d_{m0-1} = d_{m0}/r[m0]
    for m0 in pivot:-1:1
        d_nm0[m0] = d_nm0[m0+1] / r[m0]
    end
    # 上側: d_{m0+1} = r[m0+1]*d_{m0}
    for m0 in pivot:(l-1)
        d_nm0[m0+2] = r[m0+1] * d_nm0[m0+1]
    end

    # スケール合わせ：m0 = n の要素（つまり d_{n,n}）を direct で固定
    anchor = real(WignerD.wignerDjmn(l, n, n, 0.0, β, 0.0))
    scale = (d_nm0[n+1] == 0.0) ? 0.0 : anchor / d_nm0[n+1]
    d_nm0 .*= scale

    # 対称性で d_{m,n} に戻す：d_{m,n}=(-1)^(m-n) d_{n,m}
    out = zeros(Float64, l+1)
    for m in 0:l
        v = d_nm0[m+1]
        out[m+1] = isodd(m - n) ? -v : v
    end

    return out
end

d_fixedn_stable

In [16]:
l = 100
n = 1
beta = 1e-3
d = d_fixedn_stable(l, n, beta)

@show maximum(abs.(d))  # だいたい <= 1 のはず
@show d[1] d[n+1] d[end]

maximum(abs.(d)) = 0.9974768430385704
d[1] = 0.05018596913989026
d[n + 1] = 0.9974768430385704
d[end] = -4.723926516089261e-298


-4.723926516089261e-298

In [18]:
d

101-element Vector{Float64}:
  0.05018596913989026
  0.9974768430385704
 -0.05018101253658197
  0.0012609386525998963
 -2.111235521429368e-5
  2.6499143516876587e-7
 -2.659396945366735e-9
  2.222715523817659e-11
 -1.5912148324170428e-13
  9.959356208597e-16
 -5.5358972455390295e-18
  2.7666280407371686e-20
 -1.2555686174806263e-22
  ⋮
  1.9582929045630426e-256
 -5.0295712424874754e-260
  1.2211677116597195e-263
 -2.78917626638347e-267
  5.956373878905538e-271
 -1.1801006587401885e-274
  2.1471099938594522e-278
 -3.53764204373075e-282
  5.172207273824864e-286
 -6.497818328677178e-290
  6.613832299225379e-294
 -4.723926516089261e-298